<span style="color:#9A1A24;text-align:center">

# TÓPICOS DE MATEMÁTICA PARA SISTEMA AUTÓNOMOS

<div style="text-align: right"><i>  </i></div>

***
<center><h1> Visualizador de Trajetórias e Referenciais de Coordenadas com Reprodução de missão multirobótica </h1></center>

***
<span style="color:#000000;text-align:LEFT">


***



## GRUPO 5
> N: 1211059  Nome: Gonçalo Noronha  \
> N: 1190335  Nome: Alexandre Coelho 

# Conteúdo

- [1. Analise crítica dos objetivos definidos *versus* objetivos atingidos](#objetivo)
- [2. Justificação da relevância das bibliotecas Python selecionadas](#codigo)
- [3. Resultados / Evidências](#resultados)
- [4. Análise crítica do trabalho da equipa](#equipa)

<a id="objetivos"></a>
## 1. Analise crítica dos objetivos definidos *versus*  objetivos atingidos
- Um dos objetivos identificados na primeira fase foi a criação de uma global view em que se iria criar um referencial cartesiano que demonstrasse a navegação de três ROVs com suporte de playback. 
- Outro dos objetivos seria a criação uma Local View em que ao contrário da global view em que o zero do referêncial seria no mapa, este seria no ROV selecionado, juntamente com isto era necessário desenhar os movimentos dos outros ROVs que estejam na sua vizinhança. Como o objetivo anterior, é necessário suporte de playback.
- No caso do primeiro objeivo delinado no primeiro relatório, foi efetuada a criação de uma global view, onde é possivel ver o deslocamento dos três ROVs ao longo do tempo, com funcionalidade de playback. Nesta secção também foram implementadas quatro painéis de informações relativas à missão efetivada pelos três ROVs, o primeiro que foi um painél com as coordenadas Latitude, Longitude dos ROVs selecionados, um seguno que mostra a distância em tempo real entre pares e ROVs, em metros, outro que mostra as matrizes de rotação de cada ROV, com verificação em tempo real do seu determinante, por fim também foi implementado um painél onde mostra as velocidades lineares e angulares dos ROVs.
- No segundo objetivo requerido todos os objetivos foram alcançados juntamente com a adição de estatísticas de profundidade relativa e distância a robôs que estejam num raio de um quilómetro. Juntamente com isso está desenhada a trajetória de todos os robôs juntamente com os vetores de direção e cada robô é representado com um modelo tridimensional. 

<a id="codigo"></a>
## 2. Descrição do código implementado




A visão global está organizada na pasta global_plot, que reúne os módulos de leitura dos dados, reprodução temporal, representação das trajetórias e cálculos matemáticos. O ficheiro __main__.py integra estes módulos e define a interface e os callbacks, utilizando Dash e Plotly.

Os registos dos três robôs são carregados dos ficheiros CSV pela função load_robot_log(), em data_loader.py. Esta função utiliza Pandas para verificar as colunas obrigatórias e validar os campos numéricos antes de os dados serem utilizados.

Para reproduzir os movimentos numa escala temporal comum, prepare_replay_data(), em replay.py, ordena os registos e calcula o tempo de cada medição relativamente ao início da missão. A função get_replay_history() seleciona os registos até ao instante escolhido, sendo utilizada a última medição disponível de cada robô, sem interpolação entre amostras.

A representação geográfica é implementada em map_view.py, através de um mapa OpenStreetMap e dos componentes Scattermap do Plotly. Cada robô é identificado por uma cor, apresentando uma linha contínua para o percurso passado, um marcador para a posição atual e uma linha tracejada para o percurso futuro registado.

Para reduzir o custo de apresentação, simplify_path() limita o número de pontos desenhados nas linhas, preservando os dados completos para os restantes cálculos. A função update_trajectory_map() utiliza atualizações parciais (Patch), permitindo atualizar os percursos sem reconstruir toda a figura e mantendo o enquadramento escolhido pelo utilizador.

O enquadramento inicial é calculado por mission_camera(), considerando os limites dos percursos dos três robôs. O botão <Enquadrar missão> permite repor essa vista geral através da função build_initial_map().

A informação complementar é apresentada num painel lateral, onde é possível selecionar coordenadas, distâncias, matrizes de rotação ou velocidades. As coordenadas são preparadas em mission_table.py, incluindo o instante e a idade da medição, que corresponde à diferença entre o tempo selecionado e o tempo do registo utilizado.

As distâncias entre pares de robôs são calculadas em distances.py, utilizando a fórmula de Haversine. Este cálculo considera a latitude e a longitude das últimas posições disponíveis, obtendo uma distância horizontal aproximada em metros, sem incluir a componente vertical.

A orientação é tratada em orientation.py, onde os quaterniões são normalizados e convertidos em matrizes de rotação. A validade matemática destas matrizes é verificada através da ortogonalidade e do determinante, sendo os resultados apresentados pelos painéis construídos em orientation_view.py. O painel de velocidades apresenta as componentes lineares e angulares dos logs e a magnitude da velocidade linear.

A navegação temporal utiliza um slider e controlos de reprodução, pausa e velocidade. O callback control_replay() calcula o avanço com base no tempo real decorrido, enquanto os restantes callbacks atualizam o mapa e os painéis para o instante selecionado. Ao atingir o fim da missão, a reprodução para automaticamente.


A visão local consiste primáriamente de 3 ficheiros, third_person.css para estilo da página, third_person.html para a estrutura da janela e third_person.js para a implementação da visualização 3D.

O código para a implementação a visualização local utiliza a biblioteca Three para JavaScript.

Para este feito é necessário a criação de uma cena onde possamos colocar os objetos (Scene), uma camera para visualizar o ROV escolhido como ponto central (PerspectiveCamera) e um renderer para renderizar a cena (WebGLRenderer).

Para uma boa utilização desta visão, utilizamo damos capacidade ao utilizador para controlar a camera (OrbitControls).

Como se trata de algo num espaço 3D utiliza-se um grid com tom azul para defenir a superficie do mar (GridHelper).

O sistema dá fetch ás ao modelo e trajetorias através do main (robot_trajectories, robot_model_file) e dá load ás mesmas

O robô em si é renderizado por um modelo gratis encontrado na sketchlab feito por [moyshik](https://sketchfab.com/3d-models/weaver-vectored-8-thruster-rov-auv-a34c315bae734a2eaae97a2029c0a016), este robo é auxiliado por marcadores de eixos e direção.

Para diminuir o custo computacional da renderização do mesmo, é limitado para apenas robôs que estejam num raio de 1km do escolhido.

Tanto os marcadores de eixo e direção como os robôs no seu raio podem ser ativados ou observados no menu de contexto juntamente com uma barra de profundidade animada.

Para visualização da rota existe uma barra de playback no fundo da página que juntamente com a função animate, renderiza a cena para cada movimento do robo a cada 50ms (updatePose, advancePlayback).

Para a navegação e visualização das rotas dos robôs utiliza-se os dados de rotas importados, sendo visualizado em drawTrajectories, e navegação por updatePose.

<a id="resultados"></a>
## 3. Resultados / Evidências
Foram alcançdos todos os resultados pretendidos para o trabalho, onde é possivel, com este trabalho, ver o replay da missão tanto em perspétiva local como global, com playbak/replay.
Na parte global, é possivél, por vezes detetar bugs de visualização, onde os robôs ficam parados, mas os valores dos painéis continuam a ser atualizados em tempo real.

<a id="equipa"></a>
## 4. Análise crítica do trabalho da equipa

<div align="center">
<b>Tabela 1 - Avaliação Individual do Grupo</b>
</div>

<br>

<div style="display:flex; justify-content:center;">

| N | Nome | Ideação (%) | Código (%) | Documentação (%) | Total (%) |
|---|------|---------------:|--------------:|--------------------:|-------------:|
|1211059| ALUNO 1| 50%| 50%| 50%| 50%|
|1190335| ALUNO 2| 50%| 50%| 50%| 50%|
| | | | | | |
|---|Total:|100 %| 100 % | 100 % |100 %|

<br>

